In [0]:
%sql
CREATE OR REPLACE TABLE gold.fact_prescripciones AS
SELECT
    r.id_receta,
    r.id_paciente,
    r.id_medico,
    r.fecha_receta,

    dr.id_detalle,
    dr.id_medicamento,
    dr.dosis,
    dr.frecuencia,
    dr.duracion_dias,
    dr.cantidad,

    m.nombre,
    m.tipo,
    m.fabricante,
    m.stock,

    f.nombre_generico,
    f.ingrediente_activo,
    f.proposito,
    f.indicaciones_uso,
    f.advertencias,
    f.via_administracion,
    f.codigo_ndc

FROM bronze.receta r

INNER JOIN bronze.detalle_receta dr
    ON r.id_receta = dr.id_receta

LEFT JOIN silver.medicamento m
    ON dr.id_medicamento = m.id_medicamento

LEFT JOIN silver.medicamento_enriquecido_final f
    ON dr.id_medicamento = f.id_medicamento;

In [0]:
%sql
SELECT COUNT(*) AS total_prescripciones
FROM gold.fact_prescripciones;

In [0]:
%sql
SELECT *
FROM gold.fact_prescripciones
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id_receta) AS recetas,
    COUNT(DISTINCT id_medicamento) AS medicamentos,
    COUNT(DISTINCT id_paciente) AS pacientes,
    COUNT(DISTINCT id_medico) AS medicos
FROM gold.fact_prescripciones;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_medicamento AS
SELECT DISTINCT
    id_medicamento,
    nombre,
    tipo,
    fabricante,
    stock,
    nombre_generico,
    ingrediente_activo,
    proposito,
    indicaciones_uso,
    advertencias,
    via_administracion,
    codigo_ndc
FROM silver.medicamento_enriquecido_final;

In [0]:
%sql
SELECT COUNT(*) AS total_medicamentos
FROM gold.dim_medicamento;


In [0]:
print("actualizacion")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.medicamentos_multifuente AS
SELECT *
FROM silver.medicamentos_multifuente;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.resumen_ensayos_medicamentos AS
SELECT
    medicamento_cmi,
    SUM(total_ensayos) AS total_ensayos,
    SUM(ensayos_reclutando) AS ensayos_reclutando,
    SUM(ensayos_completados) AS ensayos_completados
FROM silver.resumen_ensayos_medicamentos
GROUP BY medicamento_cmi;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.resumen_diagnosticos_icd11 AS
SELECT
    diagnostico_cmi,
    descripcion_cmi,
    nombre_who,
    codigo_icd11,
    id_who,
    score,
    estado_mapeo
FROM silver.diagnosticos_icd11;

In [0]:
%sql
SHOW TABLES IN gold;

In [0]:
%sql
SELECT COUNT(*) AS total_medicamentos
FROM gold.medicamentos_multifuente;

In [0]:
%sql
SELECT COUNT(*) AS total_resumenes_ensayos
FROM gold.resumen_ensayos_medicamentos;

In [0]:
%sql
SELECT COUNT(*) AS total_diagnosticos
FROM gold.resumen_diagnosticos_icd11;